# Stage 3E — Reproduce C08’s 96.15% peak (C08-R1)

**One fresh experiment:** rerun the same reference CNN with **grayscale + CLAHE (`clipLimit=3.0`, `tileGridSize=(8,8)`)**, using the original **seed 42**, training/validation partition, architecture, hyperparameters, and 40-epoch maximum. Unlike the original C08, this notebook saves **two separate checkpoints**:

- `best_accuracy_model.keras` — highest validation accuracy; at ties, choose the epoch with lower validation loss.
- `best_loss_model.keras` — lowest validation loss, matching the original C08 model-selection rule.

Why? Original C08 hit **50/52 (96.15%) at epochs 10 and 13**, but the saved best-loss checkpoint came from epoch 12 (49/52, 94.23%). We can no longer recover the original epoch-10/13 weights, but we can test whether the result appears again and **retain both checkpoints this time**.

**Research integrity:** We will not read or predict on the 51 held-out test X-rays. The Stage 2 split remains frozen. This is **image-level** validation; patient independence is unverified. Reproducing one favorable validation epoch is not proof of generalization, and even with the same seed exact reruns may differ by hardware/software/runtime. Keep raw X-rays, private manifests, and model files out of public GitHub.

**Instructions:** In a fresh Colab runtime, select **Runtime → Run all**. Mount the same Drive. A completed C08-R1 run is skipped on subsequent executions; existing runs are never overwritten.


## 1. Mount the original Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Dependencies and private paths

Same seed and project paths as the original Stage 3C notebook. The new run lives in `clahe_grid_v1/C08-R1`, **not** `clahe_grid_v1/C08`.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = Path('/content/drive/MyDrive/224')
SPLIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1')
EXP_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/experiments/stage3')
EXP_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TO_NAME = {0: 'Normal', 1: 'Scoliosis', 2: 'Spondylolisthesis'}
CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
GRID_DIR = EXP_DIR / 'clahe_grid_v1'
print('Existing experiment folder:', EXP_DIR)
print('New CLAHE grid folder:', GRID_DIR)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print('CPU-only training detected. GPU can speed up the grid search.')
if not tf.config.list_physical_devices('GPU'):
    print('NOTE: No GPU detected. Training can run on CPU but will be slower.')
TRIAL_ID = 'C08-R1'
ORIGINAL_DIR = GRID_DIR / 'C08'
RUN_DIR = GRID_DIR / TRIAL_ID
CLIP_LIMIT = 3.0
TILE_GRID = (8, 8)
EPOCHS = 40
BATCH_SIZE = 16

print('Original:', ORIGINAL_DIR)
print('New run:', RUN_DIR)


## 3. Verify the same permanently frozen Stage 2 split

We validate the fingerprint and use only 235 training + 52 validation images. We **do not load test images**.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
config_path = SPLIT_DIR / 'split_config_PRIVATE.json'

assert manifest_path.is_file() and config_path.is_file(), (
    'Frozen Stage 2 split not found. Run Stage 2 first; do not generate a new split.'
)
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns), f'Missing manifest columns: {required - set(manifest.columns)}'
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['seed'] == SEED
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1, 'Grouped images cross splits.'
assert all(int(row.label) == CLASS_TO_LABEL[row.class_name] for row in manifest.itertuples(index=False))

rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
        .sort_values('relative_path')
        .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Frozen dataset fingerprint mismatch.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235 and counts['validation'].sum() == 52 and counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
# Intentional: no test_df or test image loader in this notebook.
assert set(train_df['relative_path']).isdisjoint(val_df['relative_path'])
print('Freeze verified. Development images:', len(train_df) + len(val_df))
print('Test set RESERVEd for final evaluation:', int(counts['test'].sum()))

## 4. Use the *exact* Stage 3C CLAHE preprocessing

In [ ]:
from PIL import Image, ImageOps

# OpenCV CLAHE is a per-image deterministic transform; it does not fit dataset statistics.
def preprocess_clahe(path, clip_limit, tile_grid, size=128):
    if not (0.0 < clip_limit <= 10.0):
        raise ValueError('clip_limit must be positive and at most 10 for this notebook')
    if not isinstance(tile_grid, tuple) or len(tile_grid) != 2:
        raise ValueError('tile_grid must be a (rows, cols) tuple')

    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(
            original, (size, size), method=Image.Resampling.LANCZOS,
            color=(0, 0, 0),
        )
        rgb = np.asarray(resized, dtype=np.uint8)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    enhanced = cv2.createCLAHE(
        clipLimit=float(clip_limit), tileGridSize=tuple(tile_grid)
    ).apply(gray)
    return enhanced[..., np.newaxis].astype(np.float32) / 255.0


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_tuned_partition(rows, clip_limit, tile_grid):
    X, y = [], []
    for item in rows.itertuples(index=False):
        path = DATA_ROOT / item.relative_path
        if not path.is_file() or file_sha256(path) != item.file_sha256:
            raise RuntimeError('An image changed since the frozen split; stop and review.')
        X.append(preprocess_clahe(path, clip_limit, tile_grid))
        y.append(int(item.label))
    return np.stack(X), np.asarray(y, dtype=np.int64)

# Sanity check on training image only.
example_path = DATA_ROOT / train_df.iloc[0]['relative_path']
for clip in (1.0, 2.0, 3.0):
    arr = preprocess_clahe(example_path, clip, (8, 8))
    assert arr.shape == (128, 128, 1) and arr.dtype == np.float32
    assert np.isfinite(arr).all() and 0 <= arr.min() <= arr.max() <= 1
print('CLAHE image preprocessing sanity checks passed.')


## 5. Use the *exact* Stage 3C reference CNN

In [ ]:
from tensorflow.keras import layers

def make_reference_cnn(shape):
    return tf.keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ], name='reference_cnn')

def make_model(shape):
    model = make_reference_cnn(shape)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model

# Inspect architecture using the correct one-channel input.
preview = make_model((128, 128, 1))
print('Reference CNN parameters (grayscale):', preview.count_params())
del preview
tf.keras.backend.clear_session()


## 6. Verify original C08 and its peak epochs

We read saved **metadata and training history**, not held-out test images. This confirms what we are trying to reproduce. If any configuration differs, stop rather than run an invalid comparison.


In [ ]:
original_config_path = ORIGINAL_DIR / 'configuration.json'
original_metrics_path = ORIGINAL_DIR / 'validation_metrics.json'
original_history_path = ORIGINAL_DIR / 'history.csv'
for p in (original_config_path, original_metrics_path, original_history_path):
    assert p.is_file(), f'Missing original C08 artifact: {p}'

original_config = json.loads(original_config_path.read_text(encoding='utf-8'))
original_metrics = json.loads(original_metrics_path.read_text(encoding='utf-8'))
original_history = pd.read_csv(original_history_path)

checks = {
    'trial': 'C08',
    'clip_limit': 3.0,
    'tile_grid': [8, 8],
    'seed': SEED,
    'epochs_max': EPOCHS,
    'batch_size': BATCH_SIZE,
    'dataset_fingerprint': fingerprint,
    'split_version': 'v1',
    'architecture': 'reference',
    'optimizer': 'Adam',
    'learning_rate': 3e-4,
}
for key, expected in checks.items():
    if original_config.get(key) != expected:
        raise RuntimeError(
            f'Original C08 {key} mismatch: '
            f'{original_config.get(key)!r} vs expected {expected!r}'
        )
assert original_metrics.get('test_evaluated') is False
assert len(original_history) == int(original_metrics['epochs_completed'])

previous_peak = float(original_history['val_accuracy'].max())
previous_peak_epochs = (
    original_history.index[
        np.isclose(original_history['val_accuracy'], previous_peak, atol=1e-6)
    ] + 1
).tolist()
print(f'Original C08 peak validation accuracy: {previous_peak:.2%}')
print('Original C08 peak epoch(s):', previous_peak_epochs)
print('Original C08 saved best-loss validation accuracy:',
      f"{original_metrics['val_accuracy']:.2%}")
print('Original software: TensorFlow', original_config.get('tensorflow_version'),
      ', OpenCV', original_config.get('opencv_version'))
print('Current software: TensorFlow', tf.__version__, ', OpenCV', cv2.__version__)
if abs(previous_peak - 50 / 52) > 1e-5:
    print('WARNING: Original saved history does not show 50/52. Review before interpreting.')

history_table = original_history[['val_accuracy', 'val_loss']].copy()
history_table.insert(0, 'epoch', history_table.index + 1)
display(history_table.loc[history_table['epoch'].isin([10, 12, 13])].round(5))


## 7. Save the highest-accuracy checkpoint (validation-loss tie-break)

The best-accuracy callback **stores the model weights at the winning epoch**, even if the minimum-loss epoch is different. The usual `ModelCheckpoint` independently retains the minimum-loss model.


In [ ]:
class SaveBestAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = str(filepath)
        self.best_accuracy = float('-inf')
        self.best_loss = float('inf')
        self.best_epoch = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        accuracy = float(logs['val_accuracy'])
        loss = float(logs['val_loss'])
        higher_accuracy = accuracy > self.best_accuracy + 1e-8
        same_accuracy_lower_loss = (
            abs(accuracy - self.best_accuracy) <= 1e-8
            and loss < self.best_loss - 1e-8
        )
        if higher_accuracy or same_accuracy_lower_loss:
            self.best_accuracy = accuracy
            self.best_loss = loss
            self.best_epoch = epoch + 1
            self.model.save(self.filepath)


def summarize_checkpoint(checkpoint_path, X_val, y_val):
    loaded = tf.keras.models.load_model(str(checkpoint_path))
    scores = loaded.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    predictions = np.argmax(scores, axis=1)
    cm = confusion_matrix(y_val, predictions, labels=[0, 1, 2])
    report = classification_report(
        y_val, predictions, labels=[0, 1, 2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True, zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_val, predictions)),
        'correct': int((predictions == y_val).sum()),
        'validation_images': int(len(y_val)),
        'macro_f1': float(f1_score(
            y_val, predictions, labels=[0, 1, 2], average='macro', zero_division=0
        )),
        'balanced_accuracy': float(balanced_accuracy_score(y_val, predictions)),
        'per_class_recall': {
            name: float(report[name]['recall']) for name in LABEL_TO_NAME.values()
        },
        'confusion_matrix': cm.tolist(),
    }


## 8. Train C08-R1 (append-only, no overwriting)

Training/validation settings reproduce Stage 3C C08, except for the additional best-accuracy checkpoint callback. If an existing C08-R1 run is complete and matches its recorded settings, its metrics are **reused** instead of retraining.


In [ ]:
def run_C08_R1():
    folder = RUN_DIR
    config_path = folder / 'configuration.json'
    result_path = folder / 'validation_metrics.json'
    model_loss_path = folder / 'best_loss_model.keras'
    model_acc_path = folder / 'best_accuracy_model.keras'

    expected = {
        'trial': TRIAL_ID,
        'source_trial': 'C08',
        'clip_limit': CLIP_LIMIT,
        'tile_grid': list(TILE_GRID),
        'seed': SEED,
        'epochs_max': EPOCHS,
        'batch_size': BATCH_SIZE,
        'dataset_fingerprint': fingerprint,
        'split_version': 'v1',
        'checkpoint_policy': 'best_val_loss_and_best_val_accuracy_loss_tiebreak',
    }

    if folder.exists():
        required = [config_path, result_path, model_loss_path, model_acc_path,
                    folder / 'history.csv', folder / 'epoch_log.csv']
        if not all(path.is_file() for path in required):
            raise RuntimeError(
                f'Incomplete C08-R1 folder: {folder}. Inspect manually; '
                'never delete or overwrite it automatically.'
            )
        saved_config = json.loads(config_path.read_text(encoding='utf-8'))
        for key, value in expected.items():
            if saved_config.get(key) != value:
                raise RuntimeError(f'Existing C08-R1 config differs on {key}. Stop.')
        saved_result = json.loads(result_path.read_text(encoding='utf-8'))
        assert saved_result.get('test_evaluated') is False
        print('C08-R1 already completed. Reusing saved metrics; no retraining.')
        return saved_result

    # Only training and validation: 235 + 52 images; never test.
    X_train, y_train = load_tuned_partition(train_df, CLIP_LIMIT, TILE_GRID)
    X_val, y_val = load_tuned_partition(val_df, CLIP_LIMIT, TILE_GRID)
    assert X_train.shape == (235, 128, 128, 1)
    assert X_val.shape == (52, 128, 128, 1)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = make_model(X_train.shape[1:])
    class_weight_values = compute_class_weight(
        'balanced', classes=np.arange(3), y=y_train
    )
    class_weights = {int(i): float(w) for i, w in enumerate(class_weight_values)}
    if model.count_params() != int(original_config['parameters']):
        raise RuntimeError('Architecture parameter count differs from original C08.')

    config = {
        **expected,
        'architecture': 'reference',
        'preprocessing': 'grayscale_clahe',
        'resize': [128, 128],
        'normalization': 'divide by 255.0 after CLAHE',
        'optimizer': 'Adam',
        'learning_rate': 3e-4,
        'class_weights': class_weights,
        'checkpoint_note': 'best accuracy ties resolved by lower validation loss',
        'early_stopping': 'val_loss patience=9 restore_best_weights=True',
        'lr_schedule': 'val_loss factor=0.5 patience=4 min_lr=1e-6',
        'tensorflow_version': str(tf.__version__),
        'opencv_version': str(cv2.__version__),
        'train_images': int(len(y_train)),
        'validation_images': int(len(y_val)),
        'patient_independent': False,
        'test_evaluated': False,
        'parameters': int(model.count_params()),
    }
    folder.mkdir(parents=True, exist_ok=False)
    config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')

    accuracy_callback = SaveBestAccuracy(model_acc_path)
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(model_loss_path), monitor='val_loss', save_best_only=True
        ),
        accuracy_callback,
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=9, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(folder / 'epoch_log.csv')),
    ]

    print('=== C08-R1: CLAHE clipLimit=3.0, tileGrid=(8,8) ===')
    start = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )
    elapsed = float(time.perf_counter() - start)
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(folder / 'history.csv', index=False)

    loss_result = summarize_checkpoint(model_loss_path, X_val, y_val)
    accuracy_result = summarize_checkpoint(model_acc_path, X_val, y_val)
    best_loss_epoch = int(np.argmin(history.history['val_loss']) + 1)
    peak = float(np.max(history.history['val_accuracy']))
    peak_epochs = (
        np.flatnonzero(np.isclose(history.history['val_accuracy'], peak, atol=1e-6)) + 1
    ).tolist()

    # Both checkpoints should correspond to the actual recorded validation behavior.
    assert abs(accuracy_result['accuracy'] - accuracy_callback.best_accuracy) < 1e-5
    assert abs(loss_result['accuracy'] - float(history.history['val_accuracy'][best_loss_epoch - 1])) < 1e-5
    assert accuracy_callback.best_epoch in peak_epochs

    metrics = {
        'trial': TRIAL_ID,
        'source_trial': 'C08',
        'clip_limit': CLIP_LIMIT,
        'tile_grid': '8x8',
        'peak_accuracy_observed': peak,
        'peak_accuracy_epochs': peak_epochs,
        'best_loss_epoch': best_loss_epoch,
        'min_validation_loss': float(np.min(history.history['val_loss'])),
        'best_accuracy_epoch': int(accuracy_callback.best_epoch),
        'best_accuracy_epoch_val_loss': float(accuracy_callback.best_loss),
        'loss_checkpoint': loss_result,
        'accuracy_checkpoint': accuracy_result,
        'epochs_completed': int(len(history.history['loss'])),
        'runtime_seconds': elapsed,
        'test_evaluated': False,
    }

    for selection, summary in [('loss', loss_result), ('accuracy', accuracy_result)]:
        np.save(folder / f'confusion_matrix_{selection}_validation.npy',
                np.asarray(summary['confusion_matrix']))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for metric, axis in [('loss', axes[0]), ('accuracy', axes[1])]:
        axis.plot(history.history[metric], label='train')
        axis.plot(history.history['val_' + metric], label='validation')
        axis.set_title('C08-R1 ' + metric)
        axis.set_xlabel('Epoch')
        axis.legend()
    fig.tight_layout()
    fig.savefig(folder / 'learning_curves.png', dpi=150)
    plt.show()
    plt.close(fig)

    # Save COMPLETE result last; an interrupted run is not silently called complete.
    result_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print('Saved result and both checkpoints in:', folder)
    return metrics

result = run_C08_R1()


## 9. Compare original C08 and the new two checkpoints

Both new checkpoint scores are recomputed from their saved `.keras` files. The table distinguishes **observed peak** and **saved-model accuracy**; we do not represent a historical peak as a preserved model.


In [ ]:
comparison = pd.DataFrame([
    {
        'run': 'C08 (original)',
        'selection': 'peak epoch only — weights NOT saved',
        'accuracy': previous_peak,
        'macro_f1': np.nan,
        'val_loss': np.nan,
        'epoch': ', '.join(map(str, previous_peak_epochs)),
    },
    {
        'run': 'C08 (original)',
        'selection': 'best-loss saved model',
        'accuracy': float(original_metrics['val_accuracy']),
        'macro_f1': float(original_metrics['val_macro_f1']),
        'val_loss': float(original_metrics['best_val_loss']),
        'epoch': int(original_metrics['best_epoch']),
    },
    {
        'run': TRIAL_ID,
        'selection': 'best-accuracy saved model',
        'accuracy': float(result['accuracy_checkpoint']['accuracy']),
        'macro_f1': float(result['accuracy_checkpoint']['macro_f1']),
        'val_loss': float(result['best_accuracy_epoch_val_loss']),
        'epoch': int(result['best_accuracy_epoch']),
    },
    {
        'run': TRIAL_ID,
        'selection': 'best-loss saved model',
        'accuracy': float(result['loss_checkpoint']['accuracy']),
        'macro_f1': float(result['loss_checkpoint']['macro_f1']),
        'val_loss': float(result['min_validation_loss']),
        'epoch': int(result['best_loss_epoch']),
    },
])
display(comparison.round(5))

for checkpoint_name, record in [
    ('best accuracy', result['accuracy_checkpoint']),
    ('best loss', result['loss_checkpoint']),
]:
    print(f'\n{TRIAL_ID}: {checkpoint_name} — '
          f'{record["correct"]}/52 = {record["accuracy"]:.2%}; '
          f'macro F1 = {record["macro_f1"]:.4f}')
    display(pd.DataFrame(
        record['confusion_matrix'],
        index=list(LABEL_TO_NAME.values()),
        columns=list(LABEL_TO_NAME.values()),
    ))

if result['accuracy_checkpoint']['correct'] >= 50:
    print('Reached at least 50/52 with a SAVED best-accuracy checkpoint.')
else:
    print('Did not reproduce 50/52 this run. This is valuable reproducibility evidence.')
print('Held-out 51 test images: NOT evaluated.')


## Next step: Repeatability across independent initialization seeds

If C08-R1 saves a 50/52 checkpoint, that's encouraging; if not, that is also informative. **Either way, don't tune to the held-out test set.** Next, we'll repeat the strongest preprocessing candidates at multiple seeds, predefine a checkpoint selection policy, compare validation macro F1, per-class recall, and loss, and then lock the model for one final test evaluation. A 1-image change in this 52-image validation partition changes accuracy by **1.92 percentage points**.

**Private Drive output:** `Spinal_Classification_Optimization/experiments/stage3/clahe_grid_v1/C08-R1/`.
